# DPO (Direct Preference Optimization) Loss

**难度:** Medium

实现 DPO（直接偏好优化）损失。

DPO 无需强化学习即可将语言模型与人类偏好对齐，使用配对的选中/拒绝对数概率。

**签名:** `dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1) -> Tensor`

**参数:**
- `policy_chosen_logps` — 策略模型对选中回复的对数概率 (B,)
- `policy_rejected_logps` — 策略模型对拒绝回复的对数概率 (B,)
- `ref_chosen_logps`, `ref_rejected_logps` — 参考模型的对数概率 (B,)
- `beta` — 温度缩放因子

**返回:** 标量损失

**约束:**
- `L = -log(sigmoid(beta * ((pi_c - ref_c) - (pi_r - ref_r)))).mean()`

**提示:**
1. chosen_logratios = β·(π_chosen - ref_chosen)
2. rejected_logratios = β·(π_rejected - ref_rejected)
3. loss = -log(sigmoid(chosen_logratios - rejected_logratios)).mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ SOLUTION

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    # ---- Step 1: 计算隐式奖励 ----
    # chosen_rewards = β * (log π_policy(chosen) - log π_ref(chosen))
    # 这是策略模型相对于参考模型在选中回复上的"偏好强度"
    # β 控制偏离参考模型的程度：β 小 → 允许更大偏移，β 大 → 更保守
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)

    # rejected_rewards = β * (log π_policy(rejected) - log π_ref(rejected))
    rejected_rewards = beta * (policy_rejected_logps - ref_rejected_logps)

    # ---- Step 2: 计算 DPO 损失 ----
    # 损失 = -log(sigmoid(chosen_rewards - rejected_rewards))
    # 直觉：我们希望 chosen_rewards > rejected_rewards
    # 即策略模型对选中回复的偏好强度 > 对拒绝回复的偏好强度
    # sigmoid 差值越大 → loss 越小（越接近 0）
    #
    # 使用 F.logsigmoid 而非 log(torch.sigmoid()) 是数值稳定技巧
    # logsigmoid(x) = log(1/(1+exp(-x))) = -softplus(-x)
    # 当 x 很大时，torch.sigmoid(x) ≈ 1，log(1) = 0，数值精度好
    # 当 x 很小时，torch.sigmoid(x) ≈ 0，log(0) = -inf，会出问题
    # logsigmoid 直接用 softplus 计算，全程数值稳定
    return -F.logsigmoid(chosen_rewards - rejected_rewards).mean()

In [ ]:
# Demo
chosen = torch.tensor([0.0, 0.0])
rejected = torch.tensor([-5.0, -5.0])
ref_c = torch.tensor([-1.0, -1.0])
ref_r = torch.tensor([-1.0, -1.0])
print('Loss:', dpo_loss(chosen, rejected, ref_c, ref_r, beta=0.1).item())

In [ ]:
from torch_judge import check
check('dpo_loss')

## 📝 核心思路总结

1. **DPO 的核心：用对数概率比代替 RL**：不需要训练奖励模型，直接从偏好数据优化策略
2. **log ratio = 隐式奖励**：β·(π - ref) 可以理解为策略相对于参考模型的"偏好奖励"
3. **sigmoid 的数值稳定实现**：用 logsigmoid 避免 log(sigmoid(x)) 在 x 很大时的数值问题